<a href="https://colab.research.google.com/github/vignesh-potharaj/gen-ai/blob/main/AgriGuardian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install required dependencies
!pip install -q -U google-genai pandas tabulate

import os
import pandas as pd
from google.colab import userdata
from google.genai import types, client

# Retrieve Gemini API Key from Colab Secrets
api_key = userdata.get('GEMINI_API_KEY')

# Initialize Gemini Client
ai = client.Client(api_key=api_key)

print("Setup complete! Dependencies and Gemini client initialized for AI AgriGuardian.")

Setup complete! Dependencies and Gemini client initialized for AI AgriGuardian.


In [3]:
# Create simulated IoT field sensor dataset (Perception Data)
agricultural_sensor_data = {
    "zone_id": ["FIELD_A1", "FIELD_A2", "FIELD_B1", "FIELD_B2", "FIELD_C1"],
    "crop_type": ["Wheat", "Tomato", "Corn", "Cotton", "Soybean"],
    "soil_moisture_pct": [14.5, 42.0, 28.0, 11.0, 35.0],  # Optimal: 25% - 35%
    "nitrogen_level_ppm": [18.0, 45.0, 12.0, 30.0, 50.0], # Target: >25 ppm
    "ambient_temp_c": [38.5, 24.0, 31.0, 41.2, 26.5],
    "forecasted_rain_mm": [0.0, 15.0, 2.0, 0.0, 25.0]
}

# Load into Pandas DataFrame to represent agent's perceptual state
df_agri_perception = pd.DataFrame(agricultural_sensor_data)

print("=== AI AGRIGUARDIAN PERCEPTION LAYER: INGESTED SENSOR DATA ===")
print(df_agri_perception.to_string(index=False))

=== AI AGRIGUARDIAN PERCEPTION LAYER: INGESTED SENSOR DATA ===
 zone_id crop_type  soil_moisture_pct  nitrogen_level_ppm  ambient_temp_c  forecasted_rain_mm
FIELD_A1     Wheat               14.5                18.0            38.5                 0.0
FIELD_A2    Tomato               42.0                45.0            24.0                15.0
FIELD_B1      Corn               28.0                12.0            31.0                 2.0
FIELD_B2    Cotton               11.0                30.0            41.2                 0.0
FIELD_C1   Soybean               35.0                50.0            26.5                25.0


In [4]:
# Function to categorize crop condition (Decision Layer)
def evaluate_crop_condition(row):
    if row["soil_moisture_pct"] < 15.0 and row["ambient_temp_c"] > 35.0:
        return "Critical Water Stress & Extreme Heat"
    elif row["nitrogen_level_ppm"] < 20.0:
        return "Nutrient Deficient (Low Nitrogen)"
    elif row["soil_moisture_pct"] > 40.0:
        return "Saturated Soil / Waterlogging Risk"
    else:
        return "Optimal Growth Condition"

# Function to generate farming recommendations (Action Layer)
def generate_agri_action(row):
    condition = row["crop_condition"]
    if condition == "Critical Water Stress & Extreme Heat":
        return "Action: Trigger automated drip irrigation (30mm) immediately during off-peak sun hours to avoid crop scorch."
    elif condition == "Nutrient Deficient (Low Nitrogen)":
        return "Action: Schedule top-dressing nitrogen fertilization via fertigation within 24 hours."
    elif condition == "Saturated Soil / Waterlogging Risk":
        return "Action: Halt irrigation schedule. Inspect soil drainage channels before forecasted rain."
    else:
        return "Action: Maintain baseline monitoring schedule. No manual intervention required."

# Apply Decision logic
df_agri_perception["crop_condition"] = df_agri_perception.apply(evaluate_crop_condition, axis=1)

# Apply Action logic
df_agri_perception["recommended_action"] = df_agri_perception.apply(generate_agri_action, axis=1)

print("=== AI AGRIGUARDIAN DECISION & ACTION LAYER ===")
for idx, row in df_agri_perception.iterrows():
    print(f"Zone: {row['zone_id']} ({row['crop_type']})")
    print(f"  - Soil Moisture: {row['soil_moisture_pct']}% | Temp: {row['ambient_temp_c']}°C | N-Level: {row['nitrogen_level_ppm']} ppm")
    print(f"  - Condition: {row['crop_condition']}")
    print(f"  - {row['recommended_action']}\n")

=== AI AGRIGUARDIAN DECISION & ACTION LAYER ===
Zone: FIELD_A1 (Wheat)
  - Soil Moisture: 14.5% | Temp: 38.5°C | N-Level: 18.0 ppm
  - Condition: Critical Water Stress & Extreme Heat
  - Action: Trigger automated drip irrigation (30mm) immediately during off-peak sun hours to avoid crop scorch.

Zone: FIELD_A2 (Tomato)
  - Soil Moisture: 42.0% | Temp: 24.0°C | N-Level: 45.0 ppm
  - Condition: Saturated Soil / Waterlogging Risk
  - Action: Halt irrigation schedule. Inspect soil drainage channels before forecasted rain.

Zone: FIELD_B1 (Corn)
  - Soil Moisture: 28.0% | Temp: 31.0°C | N-Level: 12.0 ppm
  - Condition: Nutrient Deficient (Low Nitrogen)
  - Action: Schedule top-dressing nitrogen fertilization via fertigation within 24 hours.

Zone: FIELD_B2 (Cotton)
  - Soil Moisture: 11.0% | Temp: 41.2°C | N-Level: 30.0 ppm
  - Condition: Critical Water Stress & Extreme Heat
  - Action: Trigger automated drip irrigation (30mm) immediately during off-peak sun hours to avoid crop scorch.

Zon

In [5]:
# Define prompt asking Gemini to explain agent architectures in agriculture
agri_agent_prompt = """
You are an AI Systems Engineer and Precision Agriculture Specialist.

Explain how 4 different classic AI Agent architectures would process and manage the agricultural IoT dataset (soil moisture, nitrogen levels, temperature, and rain forecasts):

1. **Simple Reflex Agent**: How does it execute immediate Condition-Action rules (e.g., IF moisture < 15% THEN turn on irrigation)?
2. **Goal-Based Agent**: How does it evaluate sequences of actions to reach a specific target goal (e.g., "Achieve optimal soil moisture of 30% by crop harvest while preventing water waste")?
3. **Utility-Based Agent**: How does it use a utility function to weigh complex trade-offs (e.g., balancing irrigation costs, yield maximization, and water conservation regulations)?
4. **Learning Agent**: How does it improve over time using historical weather trends, crop yield feedbacks, and soil degradation logs?

Provide two structured sections:
---
### PART 1: BEGINNER-FRIENDLY EXPLANATION (Analogies for Farmers & Field Managers)
Use simple agricultural analogies for each agent type.

---
### PART 2: EXPERT TECHNICAL COMPARISON (For AI Engineers)
Provide a structured breakdown highlighting state representation, decision mechanisms, and trade-offs for each agent architecture.
"""

# Dynamically select an active model available to your API key
all_models = [m.name.replace("models/", "") for m in ai.models.list()]
flash_candidates = [m for m in all_models if "flash" in m and "preview" not in m]
model_id = flash_candidates[0] if flash_candidates else "gemini-2.5-flash"

print(f"Executing Gemini AgriGuardian analysis using active model: {model_id}\n")

config = types.GenerateContentConfig(
    temperature=0.2,
    max_output_tokens=1500
)

# Generate response
response = ai.models.generate_content(
    model=model_id,
    contents=agri_agent_prompt,
    config=config
)

print("=== GEMINI AGENT ARCHITECTURE ANALYSIS ===")
print(response.text)

Executing Gemini AgriGuardian analysis using active model: gemini-2.5-flash



ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}